# Ingredient substitution using food embeddings
This notebook addresses *Ingredient Substitution*: a semantic similarity task designed to propose the most suitable alternative for a missing ingredient. To achieve this, we first generated an embedding for each ingredient by training two models — Word2Vec CBOW and GloVe — on a recipe dataset. The system then proposes available ingredients that share the highest embedding similarity with the missing one.

# Google Colab Setup

If you're using Google Colab, first run this cell:

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Switch to the "Ingredient_Substitution" directory (change according to your path)
%cd /content/drive/MyDrive/Intelligent_System/Ingredient_Substitution

# Install dependencies from "requirements.txt" in project main folder
!pip install -r ../requirements.txt

# Import dependencies

In [ ]:
import pandas as pd
import ast
import json
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec, KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

nltk.download('averaged_perceptron_tagger')  # For grammatical analysis (POS tagging)
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet') # Dictionary for lemmatization
nltk.download('punkt')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

# Show the full row when printing a dataframe
pd.set_option('display.max_colwidth', None)

# Constants

In [ ]:
# Path of the file containing the filter ingredients list
filter_ingredients_file = 'datasets/ingredient_list.csv'

# Path of the original recipe dataset to be used for generating the embeddings
recipes_for_embeddings_csv = 'datasets/recipes_for_embeddings.csv'

# Path of the refined recipe dataset used to train the Word2Vec model
refined_recipes_for_embeddings_csv = 'datasets/refined_recipes_for_embeddings.csv'

# Path of the refined recipe dataset used to train the GloVe model
refined_recipes_for_embeddings_txt = 'GloVe/refined_recipes_for_embeddings.txt'

# Path of Word2Vec embeddings
word2vec_embeddings_file = 'models/word2vec.model'

# Path of GloVe embeddings
glove_embeddings_file = 'models/GloVe.model'

# Path of the file containing "correct" substitutions to evaluate the embeddings
correct_substitutions_csv = 'datasets/correct_substitutions.csv'

# Path of the recipe dataset used to generate suggestions for the user
recipes_for_suggestions_csv = 'datasets/recipes_for_suggestions.csv'

# Data exploration and preprocessing

The strategy consists in training a model on a set a recipes to generate an embedding for each possible ingredient, so that similar ingredients map to similar embeddings.

To achieve this, we used [RecipeNLG](https://www.kaggle.com/datasets/saldenisov/recipenlg), a dataset containing over 2 million recipes. In particular, we focused on the "NER" column, which provides the list of ingredients used in each recipe.

In [ ]:
# Read the recipes dataset
df_recipe = pd.read_csv(recipes_for_embeddings_csv)

# Consider only "NER" column, which provides the list of ingredients used in each recipe
df_recipe = df_recipe['NER']

After doing some data exploration on the ingredients lists, we found out that:
- Some ingredients appear in both singular and plural forms (e.g. "egg" and "eggs")

In [ ]:
df_recipe.iloc[[20, 24]]

,NER
20,"[""sugar"", ""shortening"", ""eggs"", ""salt"", ""soda"", ""flour"", ""nuts"", ""bananas""]"
24,"[""ground beef"", ""tomato juice"", ""oats"", ""egg"", ""onion"", ""pepper"", ""salt""]"


- Some ingredients have inconsistent capitalization  (e.g. "Cheddar" and "cheddar")

In [ ]:
df_recipe.iloc[[1028, 23786]]

,NER
1028,"[""ground meat"", ""onion"", ""spinach"", ""mushrooms"", ""garlic"", ""salt"", ""shell macaroni"", ""Cheddar"", ""Parmesan cheese"", ""bread crumbs"", ""eggs"", ""oil""]"
23786,"[""frozen broccoli"", ""cream of mushroom soup"", ""mayonnaise"", ""eggs"", ""mozzarella cheese"", ""cheddar"", ""onion"", ""cracker crumbs""]"


- Some ingredients contain spelling errors (e.g. "eg" instead of "egg")

In [ ]:
df_recipe.iloc[[476131]]

,NER
476131,"[""flour"", ""baking powder"", ""sugar"", ""salt"", ""eg"", ""milk"", ""vegetable oil""]"


In such cases, these ingredients would map to different embeddings, which is not what we want. Therefore, data preprocessing is required. In particular, we need to:
- Lemmatize all the nouns to their singular form;
- Convert all the ingredients to lowercase;
- Keep an ingredient only if it is correctly spelled, by verifying its presence against a predefined list of valid ingredients. For this project, we used the [following list](https://github.com/ChantalMP/Exploiting-Food-Embeddings-for-Ingredient-Substitution/blob/master/data/cleaned_yummly_ingredients.json).

In [ ]:
def preprocess_recipe(recipe_ingredients_list, filter_ingredients_list):
    """
    Preprocesses "recipe_ingredients_list":
    - lemmatizing all nouns to their singular form;
    - converting each ingredient to lowercase,
    - keeping an in ingredient only if it is correctly spelled, by verifying its presence in "filter_ingredients_list".

    Parameters:
    - recipe_ingredients_list: list of ingredients used in a recipe
    - filter_ingredients_list: list of valid ingredients

    Returns:
    - preprocessed "recipe_ingredients_list"
    """

    lemmatizer = WordNetLemmatizer()
    clean_list = []

    for ingredient in recipe_ingredients_list:
        # Divide the ingredient string in single words (e.g. "chicken breasts" -> ["chicken", "breasts"])
        tokens = word_tokenize(ingredient)

        # Identify the POS (part of speech) tag for each word
        tagged_tokens = nltk.pos_tag(tokens)

        new_ingredient = []
        for word, tag in tagged_tokens:
            # First, lowercase the word (required for NLTK lemmatization)
            word = word.lower()

            # NLTK uses Penn Treebank POS tags to identify the part of speech for each word.
            # Nouns always have a POS tag starting with 'NN':
            # NN (Noun, singular), NNS (Noun, plural), NNP (Proper noun, singular), NNPS (Proper noun, plural)
            # Lemmatize the word only if it is identified as a noun
            if tag.startswith('NN'):
                # Lemmatize by explicitly specifying it as a noun (pos='n')
                word = lemmatizer.lemmatize(word, pos='n')

            # Whether the word was lemmatized or not, append it to the rest of the ingredient words
            new_ingredient.append(word)

        # Reconstruct the ingredient string, joining the words with an underscore
        new_ingredient_string = '_'.join(new_ingredient)

        # Keep the ingredient only if it exists in "filter_ingredient_list"
        if(new_ingredient_string in filter_ingredients_list):
            clean_list.append(new_ingredient_string)

    # Return the preprocessed ingredient list
    return clean_list

In [ ]:
# Read the file containing the filter ingredients list
df_filter_ingredients = pd.read_csv(filter_ingredients_file)

# Convert the filter ingredients list to a Python list
filter_ingredients_list = df_filter_ingredients['ingredients'].tolist()

# Convert the "NER" column of the original recipe dataset to a Python list
df_recipe = df_recipe.apply(ast.literal_eval)

# Preprocess the original recipe dataset
df_recipe = df_recipe.apply(preprocess_recipe, filter_ingredients_list = filter_ingredients_list)

# Remove empty rows due to preprocessing
df_recipe = df_recipe[df_recipe.map(len) > 0]

# Convert the "NER" column back to a string format suitable for saving to CSV
df_recipe = df_recipe.apply(json.dumps)

# Save the refined recipes dataset
df_recipe.to_csv(refined_recipes_for_embeddings_csv, index=False)

print(f"Task complete! File saved as: {refined_recipes_for_embeddings_csv}")

Task completed! File saved as: refined_recipes_for_embeddings.csv


In [ ]:
df_recipe_refined = pd.read_csv(refined_recipes_for_embeddings_csv)

# Convert the "NER" column of the refined recipe dataset to a Python list
df_recipe_refined['NER'] = df_recipe_refined['NER'].apply(ast.literal_eval)

# Training models to generate ingredient embeddings

Using the refined recipes dataset, we can train a model to generate embeddings for all the ingredients.
We can distinguish between two types of models:
- **Static embedding models:** They generate a single, fixed embedding for each ingredient, regardless of the recipe. In this case, to find potential substitutes for an ingredient, it is sufficient to identify the ingredients with the most similar embeddings;
- **Contextual embedding models:** They generate a different embedding for the same ingredient depending on the context, and thus depending on the specific recipe. In this case, to find potential substitutes for an ingredient, one would first need to compute the embedding of the ingredient within the specific recipe it appears in. Subsequently, this embedding must be compared against the embeddings of all the other ingredients across all their possible contexts of use, ultimately selecting the ingredients with the most similar embeddings.

To ensure faster, less intensive computation and lower memory usage, we opted for models from the first category. In particular, we selected two models:
- **Word2Vec CBOW:** since CBOW predicts a word based on its surrounding context, given two similar recipes where some use an ingredient and others use another, the model will map these to ingredients to similar embeddings.
- **GloVe:** by exploiting a co-occurrence matrix which tabulates how frequently words co-occur with one another, the model will generate similar embeddings for ingredients used in similar recipes.

Both models were trained using the following parameters:
- 15 epochs;
- Ignoring ingredients that appear less than 10 times in the refined recipe dataset (i.e. rare ingredients that would only introduce noise to the embeddings);
- Generating embeddings with a dimensionality of 100.



Let's start training Word2Vec model:

In [ ]:
# Prepare the input for Word2Vec.
# Word2Vec expects an iterable of "sentences", where each sentence is an iterable (in this case, it is a list of lists of ingredients)
sentences = df_recipe_refined['NER'].tolist()

# Train Word2Vec model
print("Training the model...")
model = Word2Vec(
    sentences=sentences,
    vector_size=100,  # Embedding dimensionality
    min_count=10,     # Ignore ingredients that appear less than "min_count" times
    epochs=15         # Number of epochs to train the model
)
print("Training complete!")

# Save the embeddings
model.save(word2vec_embeddings_file)

Training the model...
Training complete!


Now let's start training GloVe.
We used the [following implementation](https://github.com/stanfordnlp/glove), modifying the "demo.sh" file so that:
- the training is performed using the previously defined parameters;
- the training set points to a "refined_recipes_for_embeddings.txt" file, containing the refined recipe dataset in a specific format (see next cell).


In [ ]:
# Prepare the input for GloVe.
# GloVe expects a ".txt" file containing sentences, where words within each sentence are space-separated, and sentences are separated by a newline
df_recipe_refined_txt = df_recipe_refined['NER'].apply(lambda x: ' '.join(map(str, x)))

# Save the ".txt" file in the GloVe directory
df_recipe_refined_txt.to_csv(refined_recipes_for_embeddings_txt, index=False, header=False)

In [ ]:
# Build all
!cd GloVe && make

mkdir -p build


In [ ]:
# Train GloVe model:
print("Training the model...")
!cd GloVe && ./demo.sh
print("Training complete!")

Training the model...
mkdir -p build

$ build/vocab_count -min-count 10 -verbose 2 < refined_recipes_for_embeddings.txt > vocab.txt
BUILDING VOCABULARY
Processed 0 tokens.100000 tokens.200000 tokens.300000 tokens.400000 tokens.500000 tokens.600000 tokens.700000 tokens.800000 tokens.900000 tokens.1000000 tokens.1100000 tokens.1200000 tokens.1300000 tokens.1400000 tokens.1500000 tokens.1600000 tokens.1700000 tokens.1800000 tokens.1900000 tokens.2000000 tokens.2100000 tokens.2200000 tokens.2300000 tokens.2400000 tokens.2500000 tokens.2600000 tokens.2700000 tokens.2800000 tokens.2900000 tokens.3000000 tokens.3100000 tokens.3200000 tokens.3300000 tokens.3400000 tokens.3500000 tokens.3600000 tokens.3700000 tokens.3800000 tokens.3900000 tokens.4000000 tokens.4100000 tokens.4200000 tokens.4300000 tokens.4400000 tokens.4500000 tokens.4600000 tokens.4700000 tokens.4800000 tokens.4900000 tokens.5000000 tokens.5100000 tokens.5200000 tokens.5300000 tokens.5400000 tokens.5500000 tokens.5600000 token

In [ ]:
# Convert GloVe-format embeddings into Word2Vec-format, in order to use them with the Gensim library
glove_format_embeddings_file = 'GloVe/vectors.txt'
word2vec_format_embeddings_file = 'GloVe/vectors_word2vec.txt'

glove2word2vec(glove_format_embeddings_file, word2vec_format_embeddings_file)

model = KeyedVectors.load_word2vec_format(word2vec_format_embeddings_file)

# Save the embeddings
model.save(glove_embeddings_file)

**NB:** in some cases, building GloVe project or training a GloVe model might fail due to missing permissions. If this happens, run the following cell first:




In [ ]:
!chmod -R 777 GloVe/

Let's now analyze the embeddings and explore potential substitutes for a given ingredient, based on their cosine similarity:

In [ ]:
# Load the embeddings
word2vec_embeddings = Word2Vec.load(word2vec_embeddings_file).wv
glove_embeddings = KeyedVectors.load(glove_embeddings_file)

In [ ]:
def show_similar_ingredients(ingredient, embeddings, N=50):
  """
  Prints the top-N most similar ingredients to the given "ingredient", based on cosine similarity
  of their "embeddings".

  Parameters:
  - ingredient: the ingredient for which to find similar ingredients
  - embeddings: the embeddings for each ingredient
  - N: the number of top most similar ingredients to consider
  """
  if ingredient in embeddings:
    # Get the embedding of "ingredient"
    embedding = embeddings[ingredient]

    # Find the top-N most similar embeddings
    similars = embeddings.most_similar(ingredient, topn=N)

    print(f"\nTop-{N} ingredients with the most similar embeddings to the one of '{ingredient}':")
    for ingr, score in similars:
        print(f"- {ingr} (similarity: {score:.3f})")
  else:
        print(f"The ingredient '{ingredient}' is not in the vocabulary.")

In [ ]:
ingredient = "beef"

print("Displaying similar ingredients using embeddings generated by Word2Vec:")
show_similar_ingredients(ingredient, word2vec_embeddings)

print('\n')

print("Displaying similar ingredients using embeddings generated by GloVe:")
show_similar_ingredients(ingredient, glove_embeddings)

Displaying similar ingredients using embeddings generated by Word2Vec:

Top-50 ingredients with the most similar embeddings to the one of 'beef':
- chuck (similarity: 0.773)
- lean_beef (similarity: 0.737)
- meat (similarity: 0.727)
- stewing_beef (similarity: 0.715)
- beef_roast (similarity: 0.683)
- beef_stew_meat (similarity: 0.682)
- pot_roast (similarity: 0.677)
- cubed_beef (similarity: 0.667)
- steak (similarity: 0.659)
- stew_meat (similarity: 0.653)
- rump_roast (similarity: 0.652)
- roast (similarity: 0.650)
- chuck_roast (similarity: 0.649)
- round_steak (similarity: 0.642)
- beef_shoulder (similarity: 0.642)
- venison (similarity: 0.638)
- sirloin (similarity: 0.634)
- venison_steak (similarity: 0.634)
- chuck_steak (similarity: 0.631)
- beef_stew (similarity: 0.629)
- beef_round (similarity: 0.625)
- beef_sirloin (similarity: 0.616)
- beef_steak (similarity: 0.613)
- beef_rump (similarity: 0.598)
- beef_stock (similarity: 0.598)
- corned_beef (similarity: 0.593)
- sirloin_

# Evaluate ingredient embeddings

There is no standard method in the literature to evaluate the quality of embeddings.

In this work, we adopted the following approach:
1.   After extracting a sample of 30 ingredients, for each item in the sample, we determined what we considered to be the 5 best substitutions, resulting in a total of 150 possible "correct" substitutions;
2.   Using the developed embedding sets, for each item in the sample, we retrieved the top-*N* most similar ingredients;
3.   For each embedding set, we calculated the Precision@*N* in proposing a substitute ingredient, defined as the ratio between the number of correct substitutions found in the top-*N* list and the total number of possible correct substitutions (150, in our case).

This evaluation approach is, of course, highly subjective and heavily dependent on the gastronomic critical skills of those defining the substitutions (which, in our case, are quite limited, as our most significant culinary experience comes from the university canteen!); nevertheless, it serves as a good starting point.

In [ ]:
def precision(embeddings, correct_substitutions_csv, N):
    """
    Calculates the precision@N in proposing a substitute ingredient using the provided "embeddings".

    Precision@N is defined as the ratio between the number of times a correct substitution is present in the top-N most similar ingredients
    and the total number of possible correct substitutions.

    Correct substitutions are defined in the "correct_substitutions_csv" file.

    Parameters:
    - embeddings: the embeddings for each ingredient
    - N: the number of top most similar ingredients to consider
    - correct_substitutions_csv: path to the CSV file containing the possible correct substitutions

    Returns:
    - precision@N score
    """

    # Read the file containing possible correct substitutions
    df = pd.read_csv(correct_substitutions_csv)

    # Read the total number of possible correct substitutions
    total_entries = len(df)
    hits = 0

    # Iterate through each "ingredient-correct substitution" pair
    for index, row in df.iterrows():
        ingredient = row['ingredient']
        substitution = row['substitution']

        # Retrieve the top-N most similar ingredients to the current ingredient
        top_n = embeddings.most_similar(ingredient, topn=N)

        # Extract only ingredient names
        top_n_words = [word for word, score in top_n]

        # Check if the correct substitution is within the top-N most similar ingredients
        if substitution in top_n_words:
            hits += 1

    # Return the final precision score
    return hits / total_entries

In [ ]:
# Define the values of N to test
n_values = [10, 20, 50, 100, 200]

# Initialize lists to store the results
w2v_results = []
glove_results = []

# Calculate Precision@N for each value of N
for N in n_values:
    w2v_prec = precision(word2vec_embeddings, correct_substitutions_csv, N)
    glove_prec = precision(glove_embeddings, correct_substitutions_csv, N)

    w2v_results.append(w2v_prec)
    glove_results.append(glove_prec)

# Create the dataframe with the results
df_comparison = pd.DataFrame({
    'N': n_values,
    'Precision@N (Word2Vec)': w2v_results,
    'Precision@N (GloVe)': glove_results
})

# Set 'N' as the index for a cleaner visualization
df_comparison.set_index('N', inplace=True)

In [ ]:
# Print the dataframe
df_comparison

,Precision@N (Word2Vec),Precision@N (GloVe)
N,,
10,0.173333,0.200000
20,0.213333,0.293333
50,0.360000,0.506667
100,0.506667,0.653333
200,0.620000,0.786667


Analyzing Precision@*N* for different values of *N*, we can see that the embeddings generated by GloVe outperform those generated by Word2Vec. Let's use then GloVe embeddings to determine the top-*N* most similar ingredients for a specific item.

# BACKUP (Do not execute the following cells)

In [ ]:
!pip install gensim
!pip install nltk

If you're using Google Colab, first switch to the project directory:

In [ ]:
%cd /content/drive/MyDrive/test_IS